In [ ]:
%%bash
set -euo pipefail
cd /content
if [ ! -x bin/micromamba ]; then
  curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
export MAMBA_ROOT_PREFIX=/content/chem-mamba
bin/micromamba create -y -p /content/chem-psi4 -c conda-forge --override-channels python=3.12 psi4=1.11 rdkit matplotlib scipy
/content/chem-psi4/bin/python -c 'import psi4, rdkit; print(psi4.__version__, rdkit.__version__)'

水分子的單點能與幾何最佳化。直接使用原文座標，沒有隨機步驟。

選擇 Colab 2026.07 的 CPU 執行階段，由上往下執行。安裝格只在目前的雲端工作階段建立獨立環境。

In [ ]:
from pathlib import Path
import os
import subprocess

python = "/content/chem-psi4/bin/python"
def run(script):
    # 子程序只存圖，不沿用 Colab 核心的 inline 顯示後端。
    env = dict(os.environ, MPLBACKEND="Agg")
    result = subprocess.run([python, "-c", script], capture_output=True, text=True, env=env)
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    result.check_returncode()

### 計算

摘要會印在下方。完整的 Psi4 輸出保存在 05-02-psi4-first-calculation.log，可以從工作階段的檔案面板開啟。

In [ ]:
run(r'''
import psi4
psi4.core.set_output_file("05-02-psi4-first-calculation.log", False)
psi4.set_num_threads(2)
psi4.set_memory("1 GB")
water = psi4.geometry("""
0 1
O 0  0.0 -0.11
H 0 -1.4  1.20
H 0  1.2  1.20
units angstrom
symmetry c1
""")
level = "B3LYP/6-31G(d)"
initial = psi4.energy(level, molecule=water)
relaxed = psi4.optimize(level, molecule=water)
print(f"Input: {initial:.8f} Eh; optimized: {relaxed:.8f} Eh")
print(water.save_string_xyz_file())
''')

### 畫出最佳化前的起始幾何
圖上標出原始鍵長。用最佳化列印的 XYZ 再算一次距離，即可檢驗結構改變。

In [ ]:
run(r'''
from pathlib import Path
import matplotlib.pyplot as plt
figdir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figdir.mkdir(parents=True, exist_ok=True)
# 依作者 2024 年 8 月文章列出的輸入，計算兩個 O–H 距離
from math import hypot
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for y in (-1.4, 1.2):
    ax.plot([0, y], [-0.11, 1.2], color="#315c70", lw=3)
    ax.scatter(y, 1.2, s=420, color="#e6edf1", edgecolor="#315c70", zorder=3)
    ax.text(y, 1.2, "H", ha="center", va="center")
    ax.text(y * 0.66, 0.44, f"{hypot(y, 1.31):.2f} Å", ha="center")
ax.scatter(0, -0.11, s=500, color="#c44938", zorder=3)
ax.text(0, -0.11, "O", ha="center", va="center", color="white")
ax.set(xlim=(-1.9, 1.7), ylim=(-0.48, 1.65), xlabel="y (Å)", ylabel="z (Å)", aspect="equal", title="Water · input geometry before optimization")
fig.tight_layout()
fig.savefig(figdir / "05-02-water-input.png", dpi=180)
''')

In [ ]:
from IPython.display import Image, display
figdir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figdir / "05-02-water-input.png")))